### Preprocessing

In [1]:
import os

# Define the base path
base_path = os.path.expanduser("~/Desktop/LPU/Capstone/animal_behaviour_dataset/Videos")

# Define animal behaviors
animal_behaviors = {
    "Bear": "walking",
    "Bull": "grazing",
    "Camel": "walking",
    "Cheetah": "resting",
    "Deer": "grazing",
    "Fox": "walking",
    "Giraffe": "eating",
    "Hippopotamus": "resting",
    "Jaguar": "resting",
    "Kangaroo": "hopping",
    "Koala": "eating",
    "Leopard": "resting",
    "Lion": "resting",
    "Monkey": "sitting",
    "Mule": "walking",
    "Owl": "perching",
    "PolarBear": "walking",
    "Rabbit": "eating",
    "RedPanda": "eating",
    "Panda": "eating",
    "Rhinoceros": "grazing",
    "Snake": "slithering",
    "Squirrel": "foraging",
    "Tiger": "resting",
    "Turtle": "walking",
    "Zebra": "grazing"
}

# Loop through each animal folder
for animal, behavior in animal_behaviors.items():
    animal_folder = os.path.join(base_path, animal)
    if not os.path.exists(animal_folder):
        print(f"[!] Folder not found: {animal_folder}")
        continue

    video_files = sorted([f for f in os.listdir(animal_folder) if f.lower().endswith(".mp4")])
    
    for idx, filename in enumerate(video_files, start=1):
        new_filename = f"{animal.lower()}_{behavior}_{idx}.mp4"
        old_path = os.path.join(animal_folder, filename)
        new_path = os.path.join(animal_folder, new_filename)
        
        os.rename(old_path, new_path)
        print(f"Renamed: {filename} → {new_filename}")

print("\n✅ All videos renamed successfully!")


Renamed: Brown Bear bear nature,no copyright, creative commons license.mp4 → bear_walking_1.mp4
Renamed: _Brown Bear Walking in Water_ _Royalty Free Video Footage_.mp4 → bear_walking_2.mp4
Renamed: videoplayback (2).mp4 → bear_walking_3.mp4
Renamed: African Buffaloes (Syncerus caffer) Grazing in a Savannah.mp4 → bull_grazing_1.mp4
Renamed: Grazing Cape Buffalo.mp4 → bull_grazing_2.mp4
Renamed: videoplayback (1).mp4 → bull_grazing_3.mp4
Renamed: Camels walking in the desert.mp4 → camel_walking_1.mp4
Renamed: videoplayback (1).mp4 → camel_walking_2.mp4
Renamed: videoplayback (2).mp4 → camel_walking_3.mp4
Renamed: 20 Interesting Facts About Cheetahs - Cheetah Creative Commons Videos.mp4 → cheetah_resting_1.mp4
Renamed: Cheetah walking - 4K clip.mp4 → cheetah_resting_2.mp4
Renamed: cheetah resting in the savanna.mp4 → cheetah_resting_3.mp4
Renamed: Deer Grazing in Forest (1920x1080, 24fps) Free Creative Commons YouTube Stock Footage.mp4 → deer_grazing_1.mp4
Renamed: Deer in Forest No Copyr

### 1. Extract Features Using YOLOv8
Use your trained YOLOv8 model to track animals in the 78 normal behavior videos and extract features (speed, direction) for each tracked animal.

In [4]:
from ultralytics import YOLO
import cv2
import numpy as np
import os
import json

# Define the feature calculation function first
def calculate_features(prev_boxes, curr_boxes, prev_ids, curr_ids, prev_classes, curr_classes, time_delta):
    features = {}
    for curr_id, curr_box, curr_class in zip(curr_ids, curr_boxes, curr_classes):
        if curr_id in prev_ids and curr_class != 26:  # Skip poacher class (26)
            prev_idx = list(prev_ids).index(curr_id)
            prev_box = prev_boxes[prev_idx]
            # Calculate centroids
            prev_x, prev_y = (prev_box[0] + prev_box[2]) / 2, (prev_box[1] + prev_box[3]) / 2
            curr_x, curr_y = (curr_box[0] + curr_box[2]) / 2, (curr_box[1] + curr_box[3]) / 2
            # Speed = distance / time
            distance = np.sqrt((curr_x - prev_x)**2 + (curr_y - prev_y)**2)
            speed = distance / time_delta
            # Direction = angle in degrees
            direction = np.arctan2(curr_y - prev_y, curr_x - prev_x) * 180 / np.pi
            features[curr_id] = {"class": curr_class, "speed": speed, "direction": direction}
    return features

# Load your fine-tuned YOLOv8n model
model_path = r"C:\Users\rashi\Desktop\LPU\Capstone\best2.pt"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found at {model_path}")
model = YOLO(model_path)
behavior_data = {}
base_path = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\Videos"

# Verify base path exists
if not os.path.exists(base_path):
    raise FileNotFoundError(f"Base directory not found at {base_path}")

# Iterate through the 26 animal class folders
for animal_class in os.listdir(base_path):
    class_folder = os.path.join(base_path, animal_class)
    if not os.path.isdir(class_folder):
        continue
    print(f"Processing folder: {animal_class}")
    for video_file in os.listdir(class_folder):
        if video_file.endswith(".mp4"):
            video_path = os.path.join(class_folder, video_file)
            print(f"Processing video: {video_file}")
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                print(f"Failed to open video: {video_path}")
                continue
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps == 0:
                print(f"Invalid FPS for {video_file}, skipping...")
                cap.release()
                continue
            time_delta = 1 / fps
            prev_boxes, prev_ids, prev_classes = None, None, None
            frame_count = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                frame_count += 1
                # Track animals using YOLOv8
                results = model.track(frame, tracker="bytetrack.yaml", persist=True)
                if results[0].boxes is None or results[0].boxes.id is None:
                    print(f"Frame {frame_count} in {video_file}: No detections or tracking IDs")
                    continue
                curr_boxes = results[0].boxes.xyxy.cpu().numpy()
                curr_ids = results[0].boxes.id.cpu().numpy()
                curr_classes = results[0].boxes.cls.cpu().numpy()

                # Calculate features if we have previous frame data
                if prev_boxes is not None:
                    features = calculate_features(prev_boxes, curr_boxes, prev_ids, curr_ids, prev_classes, curr_classes, time_delta)
                    for id, feat in features.items():
                        species = model.names[int(feat["class"])]
                        if id not in behavior_data:
                            behavior_data[id] = {"class": species, "speeds": [], "directions": []}
                        behavior_data[id]["speeds"].append(feat["speed"])
                        behavior_data[id]["directions"].append(feat["direction"])
                        print(f"Frame {frame_count}: ID {id} ({species}): Speed={feat['speed']:.2f}, Direction={feat['direction']:.2f}")

                prev_boxes, prev_ids, prev_classes = curr_boxes, curr_ids, curr_classes
            cap.release()
            print(f"Finished processing {video_file}: {frame_count} frames analyzed")

# Save features to a JSON file with absolute path
output_path = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json"
try:
    with open(output_path, "w") as f:
        json.dump(behavior_data, f)
    print(f"Feature extraction complete. Features saved to {output_path}")
except Exception as e:
    print(f"Error saving JSON file: {e}")

Processing folder: Bear
Processing video: bear_walking_1.mp4

0: 384x640 1 PolarBear, 142.4ms
Speed: 8.3ms preprocess, 142.4ms inference, 9.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 PolarBear, 61.7ms
Speed: 2.0ms preprocess, 61.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Frame 2: ID 1.0 (PolarBear): Speed=0.78, Direction=-55.84

0: 384x640 1 Squirrel, 54.5ms
Speed: 2.0ms preprocess, 54.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Frame 3: ID 1.0 (Squirrel): Speed=17.76, Direction=174.17

0: 384x640 1 PolarBear, 56.0ms
Speed: 1.0ms preprocess, 56.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Frame 4: ID 1.0 (PolarBear): Speed=344.98, Direction=-172.91

0: 384x640 1 PolarBear, 71.5ms
Speed: 1.0ms preprocess, 71.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Frame 5: ID 1.0 (PolarBear): Speed=100.05, Direction=162.29

0: 384x640 1 PolarBear, 55.5ms
Speed: 2.0ms preprocess,

KeyboardInterrupt: 

### Base Technique

In [2]:
from ultralytics import YOLO
import cv2
import numpy as np
import os
import json
from pathlib import Path
from tqdm import tqdm

# Define the feature calculation function
def calculate_features(prev_boxes, curr_boxes, prev_ids, curr_ids, prev_classes, curr_classes, time_delta):
    features = {}
    for curr_id, curr_box, curr_class in zip(curr_ids, curr_boxes, curr_classes):
        curr_id = int(curr_id)  # Convert NumPy type to native Python int
        if curr_id in prev_ids and curr_class != 26:  # Skip poacher class (26)
            prev_idx = list(prev_ids).index(curr_id)
            prev_box = prev_boxes[prev_idx]
            prev_x, prev_y = (prev_box[0] + prev_box[2]) / 2, (prev_box[1] + prev_box[3]) / 2
            curr_x, curr_y = (curr_box[0] + curr_box[2]) / 2, (curr_box[1] + curr_box[3]) / 2
            distance = np.sqrt((curr_x - prev_x)**2 + (curr_y - prev_y)**2)
            speed = distance / time_delta if time_delta > 0 else 0
            direction = np.arctan2(curr_y - prev_y, curr_x - prev_x) * 180 / np.pi
            features[curr_id] = {"class": int(curr_class), "speed": float(speed), "direction": float(direction)}
    return features

# Paths
model_path = r"C:\Users\rashi\Desktop\LPU\Capstone\best2.pt"
base_path = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\Videos"
output_path = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json"
progress_file = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt"
tracker_config = "bytetrack.yaml"  # Ensure this file exists

# Verify paths exist
for path, desc in [(model_path, "Model file"), (base_path, "Base directory"), (Path(output_path).parent, "Output directory")]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{desc} not found at {path}")
print(f"All paths verified: {model_path}, {base_path}, {output_path}")

# Load the YOLOv8 model with verbose turned off
print("Loading YOLOv8 model...")
try:
    model = YOLO(model_path)
    model.verbose = False  # Disable YOLO's per-frame logging
    class_names = model.names
    print(f"Model loaded successfully with {len(class_names)} classes.")
except Exception as e:
    raise RuntimeError(f"Failed to load YOLO model: {e}")

# Load existing behavior data
behavior_data = {}
if os.path.exists(output_path):
    try:
        with open(output_path, "r") as f:
            behavior_data = json.load(f)
        print(f"Loaded existing behavior data with {len(behavior_data)} entries.")
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON, starting fresh: {e}")

# Load processed videos list
processed_videos = set()
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        processed_videos = set(f.read().splitlines())
    print(f"Loaded {len(processed_videos)} processed videos.")

# Process videos
for animal_class in os.listdir(base_path):
    class_folder = os.path.join(base_path, animal_class)
    if not os.path.isdir(class_folder):
        continue
    for video_file in os.listdir(class_folder):
        if not video_file.endswith(".mp4"):
            continue
        video_path = os.path.join(class_folder, video_file)
        if video_path in processed_videos:
            continue

        # Extract expected animal name from filename (first word before underscore or space)
        expected_animal = video_file.split("_")[0].lower()  # e.g., "bear" from "bear_walking_1.mp4"

        # Open video
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps <= 0 or total_frames <= 0:
            cap.release()
            continue
        time_delta = 1 / fps

        prev_boxes, prev_ids, prev_classes = None, None, None
        frame_count = 0

        # Progress bar for this specific video
        with tqdm(total=total_frames, desc=f"Processing {video_file}", unit="frame") as pbar:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                frame_count += 1

                # Track objects
                try:
                    results = model.track(frame, tracker=tracker_config, persist=True, verbose=False)
                    if not results or not results[0].boxes or not results[0].boxes.id:
                        pbar.update(1)
                        continue
                    curr_boxes = results[0].boxes.xyxy.cpu().numpy()
                    curr_ids = results[0].boxes.id.cpu().numpy()
                    curr_classes = results[0].boxes.cls.cpu().numpy()
                except Exception:
                    pbar.update(1)
                    continue

                # Calculate features
                if prev_boxes is not None:
                    features = calculate_features(prev_boxes, curr_boxes, prev_ids, curr_ids, prev_classes, curr_classes, time_delta)
                    for id, feat in features.items():
                        detected_species = class_names[feat["class"]].lower()  # Convert to lowercase for matching
                        # Only add data if detected species matches expected animal from filename
                        if detected_species == expected_animal:
                            if id not in behavior_data:
                                behavior_data[id] = {"class": detected_species, "speeds": [], "directions": []}
                            behavior_data[id]["speeds"].append(feat["speed"])
                            behavior_data[id]["directions"].append(feat["direction"])

                prev_boxes, prev_ids, prev_classes = curr_boxes, curr_ids, curr_classes
                pbar.update(1)

        cap.release()

        # Save progress and add completion comment
        try:
            with open(output_path, "w") as f:
                json.dump(behavior_data, f, indent=4)
            with open(progress_file, "a") as f:
                f.write(f"{video_path}\n")
            processed_videos.add(video_path)
            print(f"Processing of '{video_file}' completed. Information updated in '{output_path}' and added to '{progress_file}'.")
        except Exception as e:
            print(f"Error saving progress for {video_file}: {e}")

print(f"Feature extraction complete! All data saved to {output_path}")

All paths verified: C:\Users\rashi\Desktop\LPU\Capstone\best2.pt, C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\Videos, C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json
Loading YOLOv8 model...
Model loaded successfully with 27 classes.
Loaded existing behavior data with 25 entries.
Loaded 74 processed videos.


Processing turtle_walking_3.mp4: 100%|██████████| 1730/1730 [03:12<00:00,  8.98frame/s]


Processing of 'turtle_walking_3.mp4' completed. Information updated in 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json' and added to 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt'.


Processing zebra_grazing_1.mp4: 100%|██████████| 725/725 [02:12<00:00,  5.47frame/s]


Processing of 'zebra_grazing_1.mp4' completed. Information updated in 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json' and added to 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt'.


Processing zebra_grazing_2.mp4: 100%|██████████| 3309/3309 [05:20<00:00, 10.32frame/s]


Processing of 'zebra_grazing_2.mp4' completed. Information updated in 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json' and added to 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt'.


Processing zebra_grazing_3.mp4: 100%|██████████| 770/770 [01:15<00:00, 10.17frame/s]

Processing of 'zebra_grazing_3.mp4' completed. Information updated in 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json' and added to 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt'.
Feature extraction complete! All data saved to C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json


### Second way

In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import os
import json
from pathlib import Path
from tqdm import tqdm

# Define the feature calculation function
def calculate_features(prev_boxes, curr_boxes, prev_ids, curr_ids, prev_classes, curr_classes, time_delta):
    features = {}
    for curr_id, curr_box, curr_class in zip(curr_ids, curr_boxes, curr_classes):
        curr_id = int(curr_id)  # Convert NumPy type to native Python int
        if curr_id in prev_ids and curr_class != 26:  # Skip poacher class (26)
            prev_idx = list(prev_ids).index(curr_id)
            prev_box = prev_boxes[prev_idx]
            prev_x, prev_y = (prev_box[0] + prev_box[2]) / 2, (prev_box[1] + prev_box[3]) / 2
            curr_x, curr_y = (curr_box[0] + curr_box[2]) / 2, (curr_box[1] + curr_box[3]) / 2
            distance = np.sqrt((curr_x - prev_x)**2 + (curr_y - prev_y)**2)
            speed = distance / time_delta if time_delta > 0 else 0
            direction = np.arctan2(curr_y - prev_y, curr_x - prev_x) * 180 / np.pi
            features[curr_id] = {"class": int(curr_class), "speed": float(speed), "direction": float(direction)}
    return features

# Paths
model_path = r"C:\Users\rashi\Desktop\LPU\Capstone\best2.pt"
base_path = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\Videos"
output_path = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json"
progress_file = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt"
lstm_data_path = r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\lstm_sequences.npy"
tracker_config = "bytetrack.yaml"  # Ensure this file exists

# Verify paths exist
for path, desc in [(model_path, "Model file"), (base_path, "Base directory"), (Path(output_path).parent, "Output directory")]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{desc} not found at {path}")
print(f"All paths verified: {model_path}, {base_path}, {output_path}")

# Load the YOLOv8 model with verbose turned off
print("Loading YOLOv8 model...")
try:
    model = YOLO(model_path)
    model.verbose = False  # Disable YOLO's per-frame logging
    class_names = model.names
    print(f"Model loaded successfully with {len(class_names)} classes.")
except Exception as e:
    raise RuntimeError(f"Failed to load YOLO model: {e}")

# Load existing behavior data
behavior_data = {}
if os.path.exists(output_path):
    try:
        with open(output_path, "r") as f:
            behavior_data = json.load(f)
        print(f"Loaded existing behavior data with {len(behavior_data)} entries.")
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON, starting fresh: {e}")

# Load processed videos list
processed_videos = set()
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        processed_videos = set(f.read().splitlines())
    print(f"Loaded {len(processed_videos)} processed videos.")

# Downsampling factor (e.g., take every 5th frame)
downsample_factor = 5

# Temporary storage for raw sequences (for LSTM)
raw_sequences = []

# Process videos
for animal_class in os.listdir(base_path):
    class_folder = os.path.join(base_path, animal_class)
    if not os.path.isdir(class_folder):
        continue
    for video_file in os.listdir(class_folder):
        if not video_file.endswith(".mp4"):
            continue
        video_path = os.path.join(class_folder, video_file)
        if video_path in processed_videos:
            continue

        # Extract expected animal name from filename (first word before underscore)
        expected_animal = video_file.split("_")[0].lower()

        # Open video
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps <= 0 or total_frames <= 0:
            cap.release()
            continue
        time_delta = 1 / fps

        prev_boxes, prev_ids, prev_classes = None, None, None
        frame_count = 0
        video_speeds = []
        video_directions = []

        # Progress bar for this specific video
        with tqdm(total=total_frames, desc=f"Processing {video_file}", unit="frame") as pbar:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                frame_count += 1

                # Track objects
                try:
                    results = model.track(frame, tracker=tracker_config, persist=True, verbose=False)
                    if not results or not results[0].boxes or not results[0].boxes.id:
                        pbar.update(1)
                        continue
                    curr_boxes = results[0].boxes.xyxy.cpu().numpy()
                    curr_ids = results[0].boxes.id.cpu().numpy()
                    curr_classes = results[0].boxes.cls.cpu().numpy()
                except Exception:
                    pbar.update(1)
                    continue

                # Calculate features and downsample
                if prev_boxes is not None and frame_count % downsample_factor == 0:
                    features = calculate_features(prev_boxes, curr_boxes, prev_ids, curr_ids, prev_classes, curr_classes, time_delta)
                    for id, feat in features.items():
                        detected_species = class_names[feat["class"]].lower()
                        if detected_species == expected_animal:
                            video_speeds.append(feat["speed"])
                            video_directions.append(feat["direction"])

                prev_boxes, prev_ids, prev_classes = curr_boxes, curr_ids, curr_classes
                pbar.update(1)

        cap.release()

        # Aggregate features for this video
        if video_speeds and video_directions:  # Only if we have data
            mean_speed = float(np.mean(video_speeds))
            std_speed = float(np.std(video_speeds))
            mean_direction = float(np.mean(video_directions))
            std_direction = float(np.std(video_directions))

            # Add to behavior_data (organized by species)
            if expected_animal not in behavior_data:
                behavior_data[expected_animal] = []
            behavior_data[expected_animal].append({
                "video": video_file,
                "mean_speed": mean_speed,
                "std_speed": std_speed,
                "mean_direction": mean_direction,
                "std_direction": std_direction
            })

            # Store raw sequences for LSTM
            raw_sequences.extend(list(zip(video_speeds, video_directions)))

        # Save progress and add completion comment
        try:
            with open(output_path, "w") as f:
                json.dump(behavior_data, f, indent=4)  # Pretty-printed for readability
            with open(progress_file, "a") as f:
                f.write(f"{video_path}\n")
            processed_videos.add(video_path)
            print(f"Processing of '{video_file}' completed. Information updated in '{output_path}' and added to '{progress_file}'.")
        except Exception as e:
            print(f"Error saving progress for {video_file}: {e}")

# Preprocess raw sequences for LSTM: Convert to fixed-length sequences and save as NumPy array
print("Preparing data for LSTM...")
sequences = []
seq_length = 100  # Fixed sequence length for LSTM
for i in range(0, len(raw_sequences) - seq_length + 1, seq_length):
    seq = np.array(raw_sequences[i:i+seq_length])
    sequences.append(seq)
sequences = np.array(sequences)  # Shape: (num_sequences, seq_length, 2)
print(f"Generated {len(sequences)} sequences of length {seq_length} for LSTM training.")
np.save(lstm_data_path, sequences)  # Save as NumPy array

print(f"Feature extraction complete! All data saved to {output_path}. LSTM sequences saved to {lstm_data_path}")

c:\Users\rashi\AppData\Local\Programs\Python\Python39\lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


All paths verified: C:\Users\rashi\Desktop\LPU\Capstone\best2.pt, C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\Videos, C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json
Loading YOLOv8 model...
Model loaded successfully with 27 classes.
Loaded existing behavior data with 11 entries.
Loaded 31 processed videos.


Processing koala_eating_2.mp4: 100%|██████████| 779/779 [02:52<00:00,  4.51frame/s]


Processing of 'koala_eating_2.mp4' completed. Information updated in 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json' and added to 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt'.


Processing koala_eating_3.mp4: 100%|██████████| 623/623 [01:35<00:00,  6.52frame/s]


...

Processing of 'zebra_grazing_1.mp4' completed. Information updated in 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json' and added to 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt'.


Processing zebra_grazing_2.mp4: 100%|██████████| 3309/3309 [04:11<00:00, 13.17frame/s]


Processing of 'zebra_grazing_2.mp4' completed. Information updated in 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json' and added to 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt'.


Processing zebra_grazing_3.mp4: 100%|██████████| 770/770 [00:52<00:00, 14.77frame/s]

Processing of 'zebra_grazing_3.mp4' completed. Information updated in 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json' and added to 'C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\processed_videos.txt'.
Preparing data for LSTM...
Generated 51 sequences of length 100 for LSTM training.
Feature extraction complete! All data saved to C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\normal_behavior_data.json. LSTM sequences saved to C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\lstm_sequences.npy


### Lets see whether the sequence.npy file is ready for training or not

In [2]:
import numpy as np

# Load the sequences
sequences = np.load(r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\lstm_sequences.npy")
print(f"Loaded sequences with shape: {sequences.shape}")

Loaded sequences with shape: (51, 100, 2)


2. Train an LSTM to learn normal behavior patterns using the extracted features for the 26 animal classes.
-  Preprocess The data

In [8]:
import numpy as np

# Load the preprocessed sequences from lstm_sequences.npy
sequences = np.load(r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\lstm_sequences.npy")
print(f"Loaded sequences with shape: {sequences.shape}")  # Should be (156, 100, 2)

# Create labels (all 0s for normal behavior)
labels = np.zeros(len(sequences))  # Shape: (156,)

# Save the data
X_normal = sequences  # Shape: (156, 100, 2)
y_normal = labels     # Shape: (156,)
np.save(r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\X_normal.npy", X_normal)
np.save(r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\y_normal.npy", y_normal)
print("Saved X_normal.npy and y_normal.npy")

Loaded sequences with shape: (51, 100, 2)
Saved X_normal.npy and y_normal.npy


-  Train LSTM

In [9]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Load the data
X_normal = np.load(r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\X_normal.npy")
y_normal = np.load(r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\y_normal.npy")
print(f"X_normal shape: {X_normal.shape}, y_normal shape: {y_normal.shape}")

# Convert to PyTorch tensors
X = torch.FloatTensor(X_normal)  # Shape: (156, 100, 2)
y = torch.FloatTensor(y_normal).unsqueeze(1)  # Shape: (156, 1)

# Create a dataset and dataloader
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

X_normal shape: (51, 100, 2), y_normal shape: (51,)


In [10]:
import torch.nn as nn

class BehaviorLSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=1):
        super(BehaviorLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)  # Output: normal (0) or abnormal (1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Use last time step
        out = self.sigmoid(out)
        return out

# Initialize the model
model = BehaviorLSTM()
criterion = nn.BCELoss()  # Binary cross-entropy loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [11]:
# Training loop
num_epochs = 35
for epoch in range(num_epochs):
    for batch_X, batch_y in dataloader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

# Save the model
torch.save(model.state_dict(), r"C:\Users\rashi\Desktop\LPU\Capstone\animal_behaviour_dataset\lstm_model.pth")
print("Model saved to lstm_model.pth")

Epoch 1/35, Loss: 0.6721
Epoch 2/35, Loss: 0.6033
Epoch 3/35, Loss: 0.5653
Epoch 4/35, Loss: 0.5116
Epoch 5/35, Loss: 0.4838
Epoch 6/35, Loss: 0.4373
Epoch 7/35, Loss: 0.4149
Epoch 8/35, Loss: 0.3769
Epoch 9/35, Loss: 0.3534
Epoch 10/35, Loss: 0.3170
Epoch 11/35, Loss: 0.3075
Epoch 12/35, Loss: 0.2686
Epoch 13/35, Loss: 0.2561
Epoch 14/35, Loss: 0.2346
Epoch 15/35, Loss: 0.2040
Epoch 16/35, Loss: 0.1798
Epoch 17/35, Loss: 0.1606
Epoch 18/35, Loss: 0.1586
Epoch 19/35, Loss: 0.1415
Epoch 20/35, Loss: 0.1241
Epoch 21/35, Loss: 0.1104
Epoch 22/35, Loss: 0.1042
Epoch 23/35, Loss: 0.0944
Epoch 24/35, Loss: 0.0851
Epoch 25/35, Loss: 0.0813
Epoch 26/35, Loss: 0.0735
Epoch 27/35, Loss: 0.0653
Epoch 28/35, Loss: 0.0554
Epoch 29/35, Loss: 0.0538
Epoch 30/35, Loss: 0.0481
Epoch 31/35, Loss: 0.0467
Epoch 32/35, Loss: 0.0437
Epoch 33/35, Loss: 0.0367
Epoch 34/35, Loss: 0.0383
Epoch 35/35, Loss: 0.0333
Model saved to lstm_model.pth
